# Hex Maze Session Inventory 

In [1]:
import numpy as np
import pandas as pd
import datajoint as dj

import spyglass.common as sgc

# Hex maze behavior tables
from spyglass_hexmaze.hex_maze_behavior import (
    HexMazeBlock,
    HexCentroids,
    HexPositionSelection,
    HexPosition,
)
# Hex maze decode tables
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeDecodedPosition,
    HexMazeDecodedPositionHex,
    HexMazeDecodedHexPath,
)

/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-07-14 14:52:27,073][INFO]: DataJoint 0.14.6 connected to scrater@lmf-db.cin.ucsf.edu:3306
/home/scrater/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core 

## Find all sessions in `HexMazeBlock`

In [2]:
# Get all sessions in HexMazeBlock
hex_maze_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_maze_sessions]

print(f'Found {len(hex_maze_sessions)} hex maze sessions:')
for s in hex_maze_sessions:
    print('   ', s)
    
# Get lab + subject_id for each hex maze session
session_lab = pd.DataFrame((sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name',
                                                           as_dict=True))

# Get task type
blocks = pd.DataFrame(HexMazeBlock.fetch('nwb_file_name', 'epoch', 'task_type', as_dict=True))
epoch_task = (
    blocks.groupby(['nwb_file_name', 'epoch'])['task_type']
    .agg(lambda x: ', '.join(sorted(set(x))))  # single task type per epoch
    .reset_index()
)

# Attach lab + subject to every epoch
epoch_task = epoch_task.merge(session_lab, on='nwb_file_name', how='left')

# Flag each epoch as barrier vs probability change
epoch_task['is_barrier'] = epoch_task['task_type'].str.contains('barrier', case=False)
epoch_task['is_prob'] = epoch_task['task_type'].str.contains('prob', case=False)

n_sessions = epoch_task['nwb_file_name'].nunique()
n_epochs = len(epoch_task)
print(f'Total hex maze sessions: {n_sessions}')
print(f'Total hex maze epochs:   {n_epochs}')
print(f'Total unique subjects:   {epoch_task["subject_id"].nunique()}')

# Breakdown by lab (sessions, epochs, subjects, task type)
lab_summary = epoch_task.groupby('lab_name').agg(
    n_subjects=('subject_id', 'nunique'),
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\n By lab: ')
print(lab_summary.to_string())

# Breakdown by subject
subj_summary = epoch_task.groupby('subject_id').agg(
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\nBy subject')
print(subj_summary.to_string())

# Breakdown by task type
print('\nEpochs by task type')
print(epoch_task['task_type'].value_counts().to_string())

Found 82 hex maze sessions:
    BraveLu20240516_.nwb
    BraveLu20240518_.nwb
    BraveLu20240519_.nwb
    BraveLu20240615_.nwb
    BraveLu20240617_.nwb
    BraveLu20240619_.nwb
    BraveLu20240622_.nwb
    IM-1478_20220719_.nwb
    IM-1478_20220720_.nwb
    IM-1478_20220724_.nwb
    IM-1478_20220725_.nwb
    IM-1478_20220726_.nwb
    IM-1478_20220727_.nwb
    IM-1594_20230725_.nwb
    IM-1594_20230726_.nwb
    IM-1594_20230727_.nwb
    IM-1594_20230728_.nwb
    IM-1830_pacquiao_20250217_.nwb
    IM-1830_pacquiao_20250224_.nwb
    IM-1830_pacquiao_20250226_.nwb
    IM-1830_pacquiao_20250227_.nwb
    IM-1830_pacquiao_20250228_.nwb
    IM-1830_pacquiao_20250407_.nwb
    IM-1830_pacquiao_20250408_.nwb
    IM-1830_pacquiao_20250411_.nwb
    IM-1830_pacquiao_20250414_.nwb
    IM-1830_pacquiao_20250417_.nwb
    IM-1844_elsa_20250408_.nwb
    IM-1844_elsa_20250414_.nwb
    IM-1844_elsa_20250418_.nwb
    IM-1844_elsa_20250423_.nwb
    IM-1844_elsa_20250424_.nwb
    IM-1844_elsa_20250425_.nwb
 

## What data exists for each epoch?

For every hex maze epoch, check whether it has:
- **position**: anything in `PositionOutput`
- **ephys**: anything in `Raw`
- **theta**: anything in `HexMazeThetaV1`
- **sorted**: anything in `SpikeSortingOutput`
- **decoded**: anything in `DecodingOutput`
- **hex decode**: anything in `HexMazeDecodedPosition`

Position, theta, decoding, and hex decode each resolve to a specific epoch. Ephys and sorting are only keyed by session, so for those a session's flag applies to
all of its epochs.

In [ ]:
import re

import spyglass.spikesorting.v1 as sgs
from spyglass.common import Raw
from spyglass.position import PositionOutput
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
# Berke lab IM-* sessions have sort groups in spikesorting v1, Frank lab sessions in v0
from spyglass.spikesorting.v0.spikesorting_recording import SortGroup as SortGroupV0
from spyglass_hexmaze.hex_maze_decoding import HexMazeThetaV1

hex_session_set = set(hex_maze_sessions)


def merge_part_rows(merge_table, attrs):
    """Fetch attrs from every part of a merge table, restricted to hex maze sessions.

    We can't use merge_fetch() here: it drops an ENTIRE part if any one of the requested
    attributes is missing. PositionOutput's parts disagree on how they identify an epoch
    (Trodes keys on interval_list_name, DLC keys on epoch) and PoseV2 has no nwb_file_name
    at all, so merge_fetch would silently skip every DLC-tracked session. Instead we ask
    each part only for the attributes it actually has.
    """
    frames = []
    for part in merge_table.parts(as_objects=True):
        have = [a for a in attrs if a in part.heading.names]
        if 'nwb_file_name' not in have:
            continue  # e.g. PoseV2 -- nothing to tie it back to a session
        frames.append(pd.DataFrame(part.fetch(*have, as_dict=True)))
    df = pd.concat(frames, ignore_index=True)
    return df[df['nwb_file_name'].isin(hex_session_set)]


# Each session's run intervals ("00_r1", "01_r1", ...) and the epoch they belong to
run_intervals = {}
for row in (sgc.TaskEpoch & session_keys).fetch(
    'nwb_file_name', 'epoch', 'interval_list_name', as_dict=True
):
    run_intervals.setdefault(row['nwb_file_name'], {})[row['interval_list_name']] = row['epoch']


def interval_epoch(nwb_file_name, interval_list_name):
    """Get the epoch an interval belongs to.

    Two naming conventions show up across position, decoding, and theta:
      - the run interval itself ("00_r1" for Berke, "07_r4" for Frank), possibly with a
        suffix -- "01_r1_noPreTrialTimes" is that run interval minus pre-trial times,
        so it still belongs to the same epoch
      - a position interval, e.g. "pos 3 valid times"
    """
    name = str(interval_list_name)

    # Run interval, exact or with a suffix
    for interval, epoch in run_intervals[nwb_file_name].items():
        if name.startswith(interval):
            return epoch

    # Position interval
    match = re.match(r'pos (\d+) valid times', name)
    return int(match.group(1)) if match else None


def sort_group_table(nwb_file_name):
    """The SortGroup table (v1 or v0) that holds this session's sort groups."""
    return sgs.SortGroup if nwb_file_name.startswith('IM-') else SortGroupV0


# ---------- Session-level: ephys, sort groups, spike sorting ----------
# Raw and SortGroup are keyed by Session, and sorting is always done per session, so each
# of these flags applies to every epoch of that session.
ephys_sessions = set((Raw & session_keys).fetch('nwb_file_name'))

# Having raw ephys is NOT the same as being ready for LFP/theta -- the theta pipeline pulls
# its electrodes from SortGroup, so a session with raw ephys but no sort groups is skipped
# entirely (e.g. BraveLu has 27 epochs of raw ephys but only 7 with sort groups).
sort_group_sessions = {
    nwb for nwb in hex_maze_sessions
    if len(sort_group_table(nwb) & {'nwb_file_name': nwb}) > 0
}

# NOTE: don't use merge_fetch('nwb_file_name') on SpikeSortingOutput! Its v1 part
# (CurationV1) is keyed by sorting_id, so we'd silently miss every v1-sorted session.
# get_restricted_merge_ids walks back to the recording selection for both v0 and v1.
sorted_sessions = {
    nwb for nwb in hex_maze_sessions
    if len(SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': nwb}, sources=['v0', 'v1'], restrict_by_artifact=False
    ))
}

# ---------- Epoch-level: position, theta, decoding, hex decode ----------
# Position: DLC parts store the epoch directly, the others store "pos {epoch} valid times"
pos = merge_part_rows(PositionOutput(), ('nwb_file_name', 'interval_list_name', 'epoch'))
position_epochs = {
    (nwb, int(epoch) if pd.notna(epoch) else interval_epoch(nwb, interval))
    for nwb, interval, epoch in zip(pos['nwb_file_name'], pos['interval_list_name'], pos['epoch'])
}

# Theta: HexMazeThetaV1 inherits from LFPBandV1, so it records the run interval it ran on
theta = pd.DataFrame(
    (HexMazeThetaV1 & session_keys).fetch(
        'nwb_file_name', 'target_interval_list_name', as_dict=True
    )
)
theta_epochs = {
    (nwb, interval_epoch(nwb, interval))
    for nwb, interval in zip(theta['nwb_file_name'], theta['target_interval_list_name'])
}

# Decoding: the selection tables key on IntervalList.proj(decoding_interval=...), so each
# decode records the interval it ran on -- which gives us the epoch
dec = merge_part_rows(DecodingOutput(), ('nwb_file_name', 'decoding_interval'))
decode_epochs = {
    (nwb, interval_epoch(nwb, interval))
    for nwb, interval in zip(dec['nwb_file_name'], dec['decoding_interval'])
}

# Hex decode: keyed by TaskEpoch, so nwb_file_name + epoch come for free
hex_dec = pd.DataFrame(
    (HexMazeDecodedPosition & session_keys).fetch('nwb_file_name', 'epoch', as_dict=True)
)
hex_decode_epochs = set(zip(hex_dec['nwb_file_name'], hex_dec['epoch']))


def epoch_flag(epoch_set):
    """True for each row of epoch_task whose (session, epoch) is in epoch_set."""
    return [
        (nwb, epoch) in epoch_set
        for nwb, epoch in zip(epoch_task['nwb_file_name'], epoch_task['epoch'])
    ]


epoch_task['has_position'] = epoch_flag(position_epochs)
epoch_task['has_ephys'] = epoch_task['nwb_file_name'].isin(ephys_sessions)
epoch_task['has_sort_group'] = epoch_task['nwb_file_name'].isin(sort_group_sessions)
epoch_task['has_theta'] = epoch_flag(theta_epochs)
epoch_task['has_sorting'] = epoch_task['nwb_file_name'].isin(sorted_sessions)
epoch_task['has_decode'] = epoch_flag(decode_epochs)
epoch_task['has_hex_decode'] = epoch_flag(hex_decode_epochs)

data_cols = ['has_position', 'has_ephys', 'has_sort_group', 'has_theta',
             'has_sorting', 'has_decode', 'has_hex_decode']

# ---------- Report by epoch ----------
n_epochs = len(epoch_task)
epoch_counts = epoch_task[data_cols].sum().astype(int)

print(f'Epochs with each data type (out of {n_epochs} epochs):')
for col in data_cols:
    print(f'  {col:16} {epoch_counts[col]:4} / {n_epochs}')

# By lab, with total epochs as the first column
lab_data = epoch_task.groupby('lab_name')[data_cols].sum().astype(int)
lab_data.insert(0, 'n_epochs', epoch_task.groupby('lab_name').size())
print('\nBy lab (epochs):')
print(lab_data.to_string())

# By subject, with total epochs as the first column
subj_data = epoch_task.groupby('subject_id')[data_cols].sum().astype(int)
subj_data.insert(0, 'n_epochs', epoch_task.groupby('subject_id').size())
print('\nBy subject (epochs):')
print(subj_data.to_string())

## Full epoch inventory

One row per hex maze epoch: lab, subject, session, epoch, and what data exists for it.

In [5]:
# One row per epoch: lab, subject, session, epoch, then the has_* data columns
epoch_inventory = epoch_task[
    ['lab_name', 'subject_id', 'nwb_file_name', 'epoch'] + data_cols
].copy()

epoch_inventory = epoch_inventory.rename(
    columns={'lab_name': 'lab', 'subject_id': 'subject', 'nwb_file_name': 'session'}
)

epoch_inventory = epoch_inventory.sort_values(
    ['lab', 'subject', 'session', 'epoch']
).reset_index(drop=True)

print(f'{len(epoch_inventory)} epochs')
display(epoch_inventory)

178 epochs


,lab,subject,session,epoch,has_position,has_ephys,has_sort_group,has_theta,has_sorting,has_decode,has_hex_decode
0,Berke Lab,IM-1478,IM-1478_20220719_.nwb,0,True,True,True,True,True,True,True
1,Berke Lab,IM-1478,IM-1478_20220720_.nwb,0,True,True,True,True,True,True,True
2,Berke Lab,IM-1478,IM-1478_20220724_.nwb,0,True,True,True,True,True,True,True
3,Berke Lab,IM-1478,IM-1478_20220725_.nwb,0,True,True,True,True,True,True,True
4,Berke Lab,IM-1478,IM-1478_20220726_.nwb,0,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
173,Loren Frank,Toby,Toby20250329_.nwb,1,True,True,False,False,False,False,False
174,Loren Frank,Toby,Toby20250329_.nwb,3,True,True,False,False,False,False,False
175,Loren Frank,Toby,Toby20250329_.nwb,5,True,True,False,False,False,False,False
176,Loren Frank,Toby,Toby20250330_.nwb,1,True,True,False,False,False,False,False
